# LoRA inference on the training photos, part 2 of 3

TripoSG + the epoch-4 LoRA + our size head, run on the 1,838 training photos (the 204 holdout photos and the 511 public-test photos are done elsewhere). This notebook does one third: 8 of the 24 shards of about 77 photos, one worker per GPU.

* **Add data:** `anhanhanhanh/f3dr-tg-infer` (code, `size.pt`, `lora.pt`) and `anhanhanhanh/f3dr-tg-raw` (the photos). **Internet on.** **Accelerator: GPU T4 x2.**
* **Time:** about 4 hours if a T4 takes about 50 s per photo (that figure is from an earlier TripoSG run; this notebook has not run on Kaggle yet).
* **Output:** `lora_train_sNN.npz` per shard (`ids`, `pts` in mm, `unit` clouds of radius 1, `radius_mm` from the size head), one log per shard, and `part2_summary.json`. The last printed line is `RUN DONE` or `RUN FAILED`.
* **Caution:** the LoRA and the size head were trained on these objects, so the outputs are in-sample: better than they would be on photos the model has not seen.
* Merge all 24 shards afterwards with `finetune/merge.py <shards> --output lora_train.npz --expect 1838`.

In [ ]:
PART = 2  # 1, 2 or 3: which third of the training photos this notebook does
ARM = "lora"

import json
import os
import subprocess
import sys
import time
from pathlib import Path

NUM_PARTS, CHUNKS = 3, 4  # 3 notebooks x 2 GPU workers x 4 chunks = 24 shards of about 77 photos
NUM_SHARDS = NUM_PARTS * 2 * CHUNKS


def part_shards(part):
    """Shards that notebook `part` (1..3) does. Shard s belongs to worker s % 6; the parts take the worker pairs (0, 1), (2, 3), (4, 5)."""
    return [s for s in range(NUM_SHARDS) if (s % (NUM_PARTS * 2)) // 2 == part - 1]


def train_photos(paths):
    """Nested layout <dataset>/train/train/<group>/<category>/<id>/rgb_industrial.png: keep the paths whose inner folder is train."""
    return [p for p in paths if p.parents[3].name == "train"]


def data_root(photos):
    """The folder predict.py needs as --data: it holds train/<group>/<category>/<id>/rgb_industrial.png (the outer train folder)."""
    return photos[0].parents[4]


def link_photos(rows, flat_dir, root):
    """Flat layout: rows are (id, image path) pairs. Make root/<image path> a symlink to flat_dir/<id>.png; return the ids with no photo."""
    missing = []
    for sid, image in rows:
        src = Path(flat_dir) / f"{sid}.png"
        if not src.exists():
            missing.append(sid)
            continue
        dst = Path(root) / image
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            dst.symlink_to(src)
    return missing

W, T = Path("/kaggle/working"), Path("/kaggle/temp")
T.mkdir(exist_ok=True)
os.environ["HF_HOME"] = str(T / "hf")  # one Hugging Face cache for the pre-download and for both workers
VENV = T / "venv"
PY = str(VENV / "bin" / "python")
t0 = time.time()
FT = DATA = SIZE = SHAPE = FLAT = None  # set by stage()


def log(msg):
    print(f"[{(time.time() - t0) / 60:6.1f} min] {msg}", flush=True)


def sh(cmd, tail=1500):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
    out = (r.stdout + r.stderr)[-tail:]
    if r.returncode:
        print(out, flush=True)
        raise RuntimeError(f"failed ({r.returncode}): {cmd[:200]}")
    return out


def find_input(name):
    """Kaggle may unpack tars in datasets; accept the file or its unpacked tree."""
    hits = sorted(Path("/kaggle/input").rglob(name))
    return hits[0] if hits else None

In [ ]:
def stage():
    code = T / "f3dr"
    if not (code / "baselines/generators/triposg/finetune/predict.py").exists():
        tar = find_input("code.tar")
        if tar is not None and tar.is_file():
            code.mkdir(exist_ok=True)
            sh(f"tar -xf {tar} -C {code}")
        else:
            src = find_input("predict.py").parents[3]  # .../code/baselines
            sh(f"mkdir -p {code} && cp -r {src} {code}/")
    ft = code / "baselines/generators/triposg/finetune"
    size, shape = find_input("size.pt"), find_input(f"{ARM}.pt")
    if size is None or shape is None:
        raise RuntimeError("size.pt or lora.pt missing: add the dataset with the code and checkpoints")
    nested = train_photos(sorted(Path("/kaggle/input").rglob("rgb_industrial.png")))
    if nested:  # <dataset>/train/train/<group>/<category>/<id>/rgb_industrial.png
        if len(nested) != 2042:  # 1,838 training + 204 holdout photos live in the same folder
            raise RuntimeError(f"expected 2042 photos, found {len(nested)}")
        data, flat = data_root(nested), None
    else:  # a tar of flat <id>.png files, unpacked to train2042/
        blob = find_input("train2042.bin")
        if blob is None:
            raise RuntimeError("no training photos under /kaggle/input: add the photos dataset")
        if not (T / "train2042").exists():
            sh(f"tar xf {blob} -C {T}")
        flat = T / "train2042"
        data = T / "f3d"
        log(f"{len(list(flat.glob('*.png')))} flat photos in {flat}")
    log(f"code {ft}, photos {'flat' if flat else 'nested'} under {flat or data}, size {size}, shape {shape}")
    return ft, data, size, shape, flat

In [ ]:
def setup():
    log(sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").strip().replace("\n", " | "))
    sh("pip install -q uv")
    sh(f"uv venv -q --python 3.12 --python-preference only-managed {VENV}")
    uvp = f"uv pip install -q --python {PY}"
    sh(f"{uvp} torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124")
    sh(f"{uvp} -r {FT}/requirements.txt setuptools wheel ninja")
    if not (T / "TripoSG").exists():
        sh(f"git clone -q https://github.com/VAST-AI-Research/TripoSG.git {T}/TripoSG")
    sh(f"git -C {T}/TripoSG checkout -q fc5c40990181e2a756c4e0b1c2f4d6b5202faf8c")
    nvcc = sh("ls -d /usr/local/cuda-12* /usr/local/cuda 2>/dev/null | head -1").strip().splitlines()[-1]
    log(sh(f"{nvcc}/bin/nvcc --version | tail -1").strip())
    sh(f"CUDA_HOME={nvcc} PATH={nvcc}/bin:$PATH TORCH_CUDA_ARCH_LIST=7.5 MAX_JOBS=4 "
       f"uv pip install --python {PY} --no-build-isolation diso", tail=600)
    log(sh(f"cd /tmp && {PY} -c \"import torch, diso; print('torch', torch.__version__, torch.cuda.device_count(), 'GPUs, diso ok')\"").strip())
    # download the pinned weights once, before two workers race for them
    sh(f"cd {FT} && {PY} -c \"import torch; from huggingface_hub import snapshot_download as s; "
       f"m = torch.load('{SIZE}', map_location='cpu', weights_only=False)['metadata']; print(s(m['model'], revision=m['revision']))\"")


def prepare_photos():
    """Make DATA / <image path> exist for every training row; with the flat photo tar, link the files into place."""
    rows_file = T / "train_rows.json"
    sh(f"cd {FT} && {PY} -c \"import json, torch; m = torch.load('{SIZE}', map_location='cpu', weights_only=False)['metadata']; "
       f"json.dump([[r['id'], r['image']] for r in m['rows'] if r['split'] == 'train'], open('{rows_file}', 'w'))\"")
    rows = json.loads(rows_file.read_text())
    if FLAT is not None:
        missing = link_photos(rows, FLAT, DATA)
    else:
        missing = [sid for sid, image in rows if not (DATA / image).exists()]
    if missing:
        raise RuntimeError(f"{len(missing)} of {len(rows)} training photos not found, for example {missing[:3]}")
    log(f"{len(rows)} training photos in place under {DATA}")

In [ ]:
def run():
    shards = part_shards(PART)
    workers = max(1, min(2, int(sh("nvidia-smi -L | wc -l").strip() or 1)))
    log(f"part {PART}: shards {shards} on {workers} GPU worker(s)")
    procs = []
    for g in range(workers):
        cmds = []
        for s in shards[g::workers]:
            out, logf = W / f"{ARM}_train_s{s:02d}.npz", W / f"{ARM}_train_s{s:02d}.log"
            cmds.append(f"[ -f {out} ] || {PY} predict.py --upstream {T / 'TripoSG'} --data {DATA} --size {SIZE} --shape {SHAPE} "
                        f"--split train --steps 50 --guidance 7 --precision fp16 --dense-depth 7 --hierarchical-depth 8 "
                        f"--shard {s} --num-shards {NUM_SHARDS} --output {out} > {logf} 2>&1 || exit 1")
        procs.append(subprocess.Popen(["bash", "-c", "; ".join(cmds)], cwd=FT,
                                      env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))))
    while any(p.poll() is None for p in procs):
        time.sleep(300)
        done = sum(sum("radius=" in line for line in open(f, errors="replace")) for f in sorted(W.glob(f"{ARM}_train_s*.log")))
        log(f"photos done so far: {done}; shard files: {len(list(W.glob(f'{ARM}_train_s*.npz')))} of {len(shards)}")
    for g, p in enumerate(procs):
        if p.returncode:
            for f in sorted(W.glob(f"{ARM}_train_s*.log"))[-2:]:
                print(f.name, open(f, errors="replace").read()[-2000:], flush=True)
            raise RuntimeError(f"GPU worker {g} failed")

In [ ]:
def validate():
    import numpy as np

    n_train = len(json.loads((T / "train_rows.json").read_text()))
    shards = part_shards(PART)
    expect = sum(1 for i in range(n_train) if i % NUM_SHARDS in set(shards))
    seen, files = set(), []
    for s in shards:
        z = np.load(W / f"{ARM}_train_s{s:02d}.npz", allow_pickle=False)
        ids = z["ids"].tolist()
        if z["pts"].shape != (len(ids), 8192, 3) or not np.isfinite(z["pts"]).all() or set(ids) & seen:
            raise RuntimeError(f"shard {s} is malformed")
        seen |= set(ids)
        files.append({"shard": s, "objects": len(ids), "radius_mm_median": float(np.median(z["radius_mm"]))})
    if len(seen) != expect:
        raise RuntimeError(f"{len(seen)} objects, expected {expect}")
    (W / f"part{PART}_summary.json").write_text(json.dumps({"part": PART, "objects": len(seen), "train_rows": n_train,
                                                              "num_shards": NUM_SHARDS, "files": files}, indent=1))
    log(f"{len(seen)} clouds in {len(shards)} shard files, all finite, ids unique")

In [ ]:
try:
    FT, DATA, SIZE, SHAPE, FLAT = stage()
    setup()
    prepare_photos()
    log(f"setup done; part {PART}")
    run()
    validate()
    log("RUN DONE")
except Exception as e:
    log(f"RUN FAILED: {e!r}")
    raise